# 🚀 VS#-15M Deep Neural Language Model Trainer (Kaggle T4 GPU)
### Continuous Anti-Memorization Fine-Tuning: Epoch 151 ➔ 200+ with Hugging Face Sync

This notebook trains the **VS#-15M** (15 Million Parameters) deep neural language model on Kaggle's free GPU (Tesla T4 or P100).

### 🧠 Overcoming Dataset Memorization (Epoch 150 Analysis):
At Epoch 150, the model achieved **99.8% accuracy and 0.0101 loss** because 15 Million parameters memorized the smaller static dataset verbatim.

**⚡ Upgrades in this New Training Phase:**
- 🔄 **Zero-Loss Warm Start**: Automatically loads your 150-epoch checkpoint from [alexon-v8/VS-Sharp](https://huggingface.co/alexon-v8/VS-Sharp) and resumes seamlessly from **Epoch 151**.
- 📚 **Vastly Expanded Dataset**: Thousands of combinatorial algorithms, polyglot tasks (`inject python`, `inject js`, `inject c`), stdlib APIs, and code repairs.
- 🛡️ **Label Smoothing (0.1)**: Replaces hard one-hot targets with soft distributions, mathematically preventing the 99.8% overconfidence trap.
- 🎲 **Dropout (p=0.15)**: Prevents co-adaptation of neurons and forces true generalized understanding.
- 📉 **Fine-Tuning Learning Rate ($1.5 \times 10^{-4}$)**: Gentle Cosine Annealing learning rate schedule preserves pre-trained knowledge without catastrophic forgetting.
- ☁️ **Auto-Sync to Hugging Face**: Automatically pushes updated weights directly back to Hugging Face Hub after training.

**⚠️ REQUIRED KAGGLE SETTINGS:**
- Right sidebar **Notebook Settings** ➔ **Accelerator**: `GPU T4 x2` (or `GPU P100`).
- Right sidebar **Notebook Settings** ➔ **Internet**: **`Internet on`**.

## 1. Verify GPU Setup

In [ ]:
!nvidia-smi

## 2. Clone Repository & Pull Epoch 150 Checkpoint from Hugging Face

In [ ]:
import os, urllib.request
%cd /kaggle/working
!rm -rf VS-Sharp
!git clone https://github.com/VerScript/VS-Sharp.git
%cd /kaggle/working/VS-Sharp

HF_URL = "https://huggingface.co/alexon-v8/VS-Sharp/resolve/main/model_weights.json"
print(f"[*] Pulling 150-epoch weights from Hugging Face: {HF_URL}...")
urllib.request.urlretrieve(HF_URL, "model_weights.json")
size_mb = os.path.getsize("model_weights.json") / (1024 * 1024)
print(f"✅ Successfully downloaded model_weights.json ({size_mb:.1f} MB)! Ready for Epoch 151 warm-start.")

## 3. Rebuild Expanded Master Dataset

In [ ]:
!node build_dataset.js

## 4. Run Anti-Memorization Fine-Tuning (Epochs 151 ➔ 200+)
Regularized training with Dropout (0.15), Label Smoothing (0.1), AdamW weight decay, and Cosine Annealing.

In [ ]:
import getpass, os
# Enter your Hugging Face write token (or set Kaggle Secret HF_TOKEN)
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    hf_token = getpass.getpass('Enter your Hugging Face Write Token: ')

!pip install -q huggingface_hub
!python train_gpu.py \
    --epochs 50 \
    --batch-size 128 \
    --samples-per-epoch 20000 \
    --lr 0.00015 \
    --dropout 0.15 \
    --label-smoothing 0.1 \
    --weight-decay 0.01 \
    --hf-token "$hf_token" \
    --hf-repo "alexon-v8/VS-Sharp"

## 5. Visualize Loss & Accuracy Curves

In [ ]:
import json
import matplotlib.pyplot as plt

if os.path.exists('epoch_eval_history.json'):
    with open('epoch_eval_history.json', 'r') as f:
        history = json.load(f)

    epochs = [h['epoch'] for h in history]
    losses = [h['loss'] for h in history]
    accuracies = [h['accuracy'] * 100 for h in history]

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, losses, 'r-o', label='Loss (Label Smoothed)')
    plt.title('VS#-15M Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracies, 'b-o', label='Accuracy')
    plt.title('VS#-15M Accuracy Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

## 6. Live Interactive Prompt Tester with Top-P Nucleus Sampling

In [ ]:
import torch, json
from train_gpu import VSSharp15M, generate_text

with open('model_weights.json', 'r') as f:
    w_data = json.load(f)

vocab = w_data['vocab']
vocab_map = {t: idx for idx, t in enumerate(vocab)}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

hidden_size_1 = w_data.get('config', {}).get('hiddenSize', 1020)
model = VSSharp15M(len(vocab), hidden_size_1).to(device)

w = w_data['weights']
model.fc1.weight.data.copy_(torch.tensor(w['W1'], dtype=torch.float32).t())
model.fc1.bias.data.copy_(torch.tensor(w['b1'], dtype=torch.float32))
model.fc2.weight.data.copy_(torch.tensor(w['W2'], dtype=torch.float32).t())
model.fc2.bias.data.copy_(torch.tensor(w['b2'], dtype=torch.float32))

old_e = torch.tensor(w['E'], dtype=torch.float32)
model.embedding.weight.data[:old_e.size(0), :].copy_(old_e)
w3_t = torch.tensor(w['W3'], dtype=torch.float32).t()
model.fc_out.weight.data[:w3_t.size(0), :].copy_(w3_t)
model.fc_out.bias.data[:len(w['b3'])].copy_(torch.tensor(w['b3'], dtype=torch.float32))

test_prompts = [
    "hello",
    "write a hello world program in verscript",
    "write a function to check if a number is prime in verscript",
    "how to fix count = 10 in verscript",
    "how to run python in verscript",
    "write javascript async fetch function",
    "write c code for binary search",
    "how to use Math.sqrt in verscript"
]

print("=== Testing VS#-15M Live Generation (Post Epoch 150) ===")
for p in test_prompts:
    res = generate_text(model, vocab, vocab_map, p, device, max_tokens=60, temperature=0.75, top_p=0.9)
    print(f"\n[Prompt]: {p}")
    print(f"[Generated Response]:\n{res}")

## 7. Direct 1-Click Browser Download (Kaggle Output)

In [ ]:
!cp /kaggle/working/VS-Sharp/model_weights.json.gz /kaggle/working/
!cp /kaggle/working/VS-Sharp/epoch_eval_history.json /kaggle/working/
print('✅ Copied weights to /kaggle/working/')
print('👉 Look at the right sidebar under Data > Output to download model_weights.json.gz directly!')